# ดึงพิกัด 32 ไซต์จากลิงก์ Google Maps + คำนวณระยะทางถึงถนนจริง (OSM)

**วัตถุประสงค์:** โน้ตบุ๊กนี้ทำ 2 อย่างต่อเนื่องกัน
1. **ดึงพิกัด lat/lon** ของ 32 ไซต์ จากลิงก์ Google Maps แบบย่อ (`goo.gl/maps/...`) ที่บันทึกไว้ในไฟล์ข้อมูลภาคสนาม (คอลัมน์ "พิกัด") — เนื่องจากไฟล์เดิมมีแค่ลิงก์ ไม่มีตัวเลขพิกัด
2. **คำนวณระยะทางถึงถนนจริง (road centerline)** จาก OpenStreetMap แทนเส้นทางโดยประมาณเดิม

**อินพุตที่ต้องเตรียม:** ไฟล์ Excel/CSV ที่มีอย่างน้อยคอลัมน์:
- ชื่อไซต์ (เช่น "จุดตรวจวัดอากาศ-1")
- ลิงก์ Google Maps (คอลัมน์ "พิกัด" หรือ `maps_url`)
- ที่อยู่ข้อความ (คอลัมน์ "ที่อยู่" หรือ `address`) — ใช้เป็นตัวสำรองถ้าลิงก์ดึงพิกัดไม่สำเร็จ

**หมายเหตุสำคัญก่อนเริ่ม:** Colab มีการเชื่อมต่ออินเทอร์เน็ตจริง จึงสามารถเปิด/ตาม-redirect ลิงก์ `goo.gl` ได้ (ต่างจากระบบแชทที่ผมใช้คุยกับคุณซึ่งเข้าถึงลิงก์เหล่านี้ไม่ได้) แต่ Google ประกาศทยอยปิดระบบ `goo.gl` ลิงก์เก่าบางส่วนอาจ redirect ไม่สำเร็จแล้ว — ถ้าไซต์ไหนดึงไม่ได้ ให้เปิดลิงก์นั้นด้วยมือจากมือถือ/เบราว์เซอร์แล้วคัดลอกพิกัดมาใส่เองในขั้นตอนตรวจสอบท้ายเซกชันที่ 1


## ขั้นตอนที่ 1: ติดตั้งไลบรารีที่จำเป็น

In [ ]:
!pip install osmnx geopandas shapely pyproj geopy --quiet
print("ติดตั้งเสร็จแล้ว")


In [ ]:
import re
import time
import requests
import pandas as pd
import numpy as np
import osmnx as ox
import geopandas as gpd
from shapely.geometry import Point
from geopy.geocoders import Nominatim
from google.colab import files
from tqdm.notebook import tqdm

print("osmnx version:", ox.__version__)


## ขั้นตอนที่ 2: อัปโหลดไฟล์ข้อมูล 32 ไซต์

อัปโหลดไฟล์ Excel/CSV ที่มีคอลัมน์ชื่อไซต์ ลิงก์ Google Maps และที่อยู่ (เช่นไฟล์บันทึกการตรวจวัดที่ใช้อยู่แล้ว)


In [ ]:
uploaded = files.upload()
data_filename = list(uploaded.keys())[0]
print(f"อัปโหลดไฟล์: {data_filename}")


In [ ]:
if data_filename.lower().endswith(('.xlsx', '.xls')):
    raw = pd.read_excel(data_filename, header=0)
else:
    raw = pd.read_csv(data_filename)

print("คอลัมน์ที่พบในไฟล์:")
print(list(raw.columns))
raw.head(5)


## ขั้นตอนที่ 3: ระบุคอลัมน์ที่เกี่ยวข้อง

แก้ชื่อคอลัมน์ในบรรทัด `SITE_COL`, `URL_COL`, `ADDRESS_COL` ด้านล่างให้ตรงกับไฟล์ของคุณ ถ้าเดาไม่ตรงให้แก้เอง


In [ ]:
def guess_col(columns, keywords):
    for c in columns:
        cl = str(c).lower()
        for kw in keywords:
            if kw in cl or kw in str(c):
                return c
    return None

SITE_COL = guess_col(raw.columns, ['site', 'id', 'name', 'จุดตรวจวัด']) or raw.columns[0]
URL_COL = guess_col(raw.columns, ['maps_url', 'url', 'link', 'พิกัด'])
ADDRESS_COL = guess_col(raw.columns, ['address', 'ที่อยู่'])

print(f"คอลัมน์ชื่อไซต์: {SITE_COL}")
print(f"คอลัมน์ลิงก์แผนที่: {URL_COL}")
print(f"คอลัมน์ที่อยู่: {ADDRESS_COL}")

sites = raw[[SITE_COL] + ([URL_COL] if URL_COL else []) + ([ADDRESS_COL] if ADDRESS_COL else [])].copy()
sites.columns = ['site_id'] + (['maps_url'] if URL_COL else []) + (['address'] if ADDRESS_COL else [])
sites = sites.dropna(subset=['site_id']).reset_index(drop=True)
print(f"จำนวนไซต์: {len(sites)}")
sites.head(10)


## ขั้นตอนที่ 4: ตาม-redirect ลิงก์ Google Maps เพื่อดึงพิกัด

ฟังก์ชันนี้จะเปิดลิงก์ `goo.gl` แล้วตามการ redirect ไปจนถึง URL เต็มของ Google Maps จากนั้นดึงพิกัดจากรูปแบบ URL สองแบบที่เป็นไปได้:
- `!3d{lat}!4d{lon}` — พิกัดของหมุด (แม่นยำที่สุด ถ้ามี)
- `@{lat},{lon},{zoom}z` — จุดกึ่งกลางแผนที่ (สำรอง ถ้าไม่มีแบบแรก)


In [ ]:
def resolve_coords_from_maps_url(url, timeout=10):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                      '(KHTML, like Gecko) Chrome/120.0 Safari/537.36'
    }
    try:
        r = requests.get(url, headers=headers, timeout=timeout, allow_redirects=True)
        final_url = r.url
    except Exception as e:
        return None, None, None, f"request_failed: {e}"

    m = re.search(r'!3d(-?\d+\.\d+)!4d(-?\d+\.\d+)', final_url)
    if m:
        return float(m.group(1)), float(m.group(2)), final_url, 'pin_3d4d'

    m = re.search(r'@(-?\d+\.\d+),(-?\d+\.\d+)', final_url)
    if m:
        return float(m.group(1)), float(m.group(2)), final_url, 'map_center_@'

    return None, None, final_url, 'no_coords_found_in_url'


In [ ]:
lats, lons, final_urls, methods = [], [], [], []

for url in tqdm(sites.get('maps_url', pd.Series([None]*len(sites))), desc="กำลังตามลิงก์"):
    if pd.isna(url):
        lats.append(None); lons.append(None); final_urls.append(None); methods.append('no_url')
        continue
    lat, lon, final_url, method = resolve_coords_from_maps_url(url)
    lats.append(lat); lons.append(lon); final_urls.append(final_url); methods.append(method)
    time.sleep(0.5)  # หน่วงเวลาเล็กน้อยเพื่อลดความเสี่ยงถูกจำกัดอัตราการเรียก (rate limit)

sites['lat'] = lats
sites['lon'] = lons
sites['resolved_url'] = final_urls
sites['coord_method'] = methods

print(sites['coord_method'].value_counts())
sites[['site_id', 'lat', 'lon', 'coord_method']]


## ขั้นตอนที่ 5: สำรอง — geocode จากที่อยู่ข้อความ สำหรับไซต์ที่ดึงพิกัดจากลิงก์ไม่สำเร็จ

⚠️ วิธีนี้ให้ความแม่นยำระดับตำบล/อำเภอเท่านั้น ไม่แม่นยำเท่าพิกัดจริงจากหมุด **ควรตรวจสอบด้วยตนเองก่อนใช้งานจริงเสมอ**


In [ ]:
geolocator = Nominatim(user_agent="saraburi_pm25_study")

need_fallback = sites['lat'].isna() & sites.get('address', pd.Series([None]*len(sites))).notna()
print(f"จำนวนไซต์ที่ต้องใช้วิธี geocode ที่อยู่สำรอง: {need_fallback.sum()}")

for idx in sites[need_fallback].index:
    addr = sites.loc[idx, 'address']
    try:
        loc = geolocator.geocode(addr + ', Thailand', timeout=10)
        if loc:
            sites.loc[idx, 'lat'] = loc.latitude
            sites.loc[idx, 'lon'] = loc.longitude
            sites.loc[idx, 'coord_method'] = 'geocoded_address_fallback_LOW_PRECISION'
    except Exception as e:
        print(f"geocode ล้มเหลวสำหรับ {sites.loc[idx, 'site_id']}: {e}")
    time.sleep(1)  # Nominatim กำหนดอัตราการเรียกไม่เกิน 1 ครั้ง/วินาที

sites[['site_id', 'lat', 'lon', 'coord_method']]


## ขั้นตอนที่ 6: ตรวจสอบก่อนใช้งานจริง (สำคัญมาก)

ตรวจดูตารางด้านล่าง:
- แถวที่ `coord_method` เป็น `no_url`, `no_coords_found_in_url`, หรือ `request_failed...` — **ยังไม่มีพิกัด** ต้องเปิดลิงก์เดิมด้วยมือแล้วกรอกพิกัดเองในเซลล์ถัดไป
- แถวที่เป็น `geocoded_address_fallback_LOW_PRECISION` — มีพิกัดแต่แม่นยำต่ำ ควรตรวจสอบเทียบกับความรู้ภาคสนามจริงก่อนนำไปใช้


In [ ]:
missing_or_low = sites[sites['lat'].isna() | (sites['coord_method'] == 'geocoded_address_fallback_LOW_PRECISION')]
print(f"จำนวนไซต์ที่ต้องตรวจสอบ/กรอกด้วยมือ: {len(missing_or_low)}")
missing_or_low[['site_id', 'lat', 'lon', 'coord_method']]


In [ ]:
# ตัวอย่างการแก้พิกัดด้วยมือ (ลบ # ออกแล้วแก้ไขค่าตามจริง หากมีไซต์ที่ต้องกรอกเอง)
# sites.loc[sites['site_id'] == 'จุดตรวจวัดอากาศ-XX', 'lat'] = 14.xxxxx
# sites.loc[sites['site_id'] == 'จุดตรวจวัดอากาศ-XX', 'lon'] = 101.xxxxx
# sites.loc[sites['site_id'] == 'จุดตรวจวัดอากาศ-XX', 'coord_method'] = 'manual_entry'


## ขั้นตอนที่ 7: บันทึกไฟล์พิกัดที่ยืนยันแล้ว (เก็บไว้ใช้ซ้ำในอนาคต)

บันทึกไฟล์นี้เก็บไว้ — ครั้งต่อไปไม่ต้องดึงพิกัดจากลิงก์ใหม่อีก อัปโหลดไฟล์นี้แทนได้เลย


In [ ]:
coords_filename = 'site_coordinates_resolved.csv'
sites[['site_id', 'lat', 'lon', 'coord_method']].to_csv(coords_filename, index=False, encoding='utf-8-sig')
files.download(coords_filename)
print(f"บันทึกไฟล์: {coords_filename}")


---
# ส่วนที่ 2: คำนวณระยะทางถึงถนนจริงจาก OpenStreetMap

จากนี้ใช้พิกัดที่ได้ในส่วนที่ 1 (ตัวแปร `sites`) ต่อเนื่องได้เลย โดยไม่ต้องอัปโหลดไฟล์ใหม่


In [ ]:
sites_clean = sites.dropna(subset=['lat', 'lon']).copy().reset_index(drop=True)
print(f"จำนวนไซต์ที่มีพิกัดพร้อมใช้งาน: {len(sites_clean)} จากทั้งหมด {len(sites)}")
if len(sites_clean) < len(sites):
    print("⚠️ ยังมีไซต์ที่ไม่มีพิกัด ระยะทางถนนจะคำนวณได้เฉพาะไซต์ที่มีพิกัดครบเท่านั้น")


## ขั้นตอนที่ 8: กำหนดขอบเขตพื้นที่และดึงโครงข่ายถนนจาก OSM

ระยะทางเดิมในต้นฉบับมีตั้งแต่ 0.4 ถึง 74.2 กม. จึงขยายกรอบพื้นที่ (buffer ~0.3 องศา ≈ 33 กม.) ให้ครอบคลุมมากพอ


In [ ]:
buffer_deg = 0.3
north = sites_clean['lat'].max() + buffer_deg
south = sites_clean['lat'].min() - buffer_deg
east = sites_clean['lon'].max() + buffer_deg
west = sites_clean['lon'].min() - buffer_deg

print(f"ขอบเขตพื้นที่ดึงข้อมูล: N={north:.3f}, S={south:.3f}, E={east:.3f}, W={west:.3f}")

custom_filter = '["highway"~"motorway|trunk|primary|secondary"]'

G = ox.graph_from_bbox(
    bbox=(north, south, east, west),
    custom_filter=custom_filter,
    simplify=True,
    retain_all=True
)
print(f"จำนวนโหนด: {len(G.nodes)}, จำนวนเส้นทาง (edges): {len(G.edges)}")


## ขั้นตอนที่ 9: แปลงพิกัดเป็นระบบ UTM (EPSG:32647) เพื่อคำนวณระยะทางเป็นเมตรอย่างแม่นยำ


In [ ]:
G_proj = ox.project_graph(G, to_crs='EPSG:32647')

sites_gdf = gpd.GeoDataFrame(
    sites_clean,
    geometry=[Point(xy) for xy in zip(sites_clean['lon'], sites_clean['lat'])],
    crs='EPSG:4326'
).to_crs('EPSG:32647')

X = sites_gdf.geometry.x.values
Y = sites_gdf.geometry.y.values
print("แปลงพิกัดเสร็จแล้ว")


## ขั้นตอนที่ 10: คำนวณระยะทางถึงถนนที่ใกล้ที่สุด (โครงข่ายทั้งหมด)


In [ ]:
nearest_edges, dists_m = ox.distance.nearest_edges(G_proj, X, Y, return_dist=True)
sites_clean['dist_to_nearest_road_km'] = np.array(dists_m) / 1000
sites_clean[['site_id', 'lat', 'lon', 'dist_to_nearest_road_km']].sort_values('dist_to_nearest_road_km')


## ขั้นตอนที่ 11: คำนวณระยะทางเฉพาะถึงถนนมิตรภาพ / ทางหลวงหมายเลข 2


In [ ]:
def is_mittraphap(data):
    name = data.get('name', '')
    ref = data.get('ref', '')
    names = name if isinstance(name, list) else [name]
    refs = ref if isinstance(ref, list) else [ref]
    names = [str(n) for n in names]
    refs = [str(r) for r in refs]
    name_match = any('mittraphap' in n.lower() or 'มิตรภาพ' in n for n in names)
    ref_match = any(r.strip() in ['2', 'AH1', 'AH 1'] for r in refs)
    return name_match or ref_match

mtr_edges = [(u, v, k) for u, v, k, data in G_proj.edges(keys=True, data=True) if is_mittraphap(data)]
print(f"จำนวนเส้นทางที่ตรงกับถนนมิตรภาพ/ทางหลวงหมายเลข 2: {len(mtr_edges)}")

if len(mtr_edges) == 0:
    print("⚠️ ไม่พบถนนที่ตรงกับเงื่อนไข ลองเปลี่ยน custom_filter ในขั้นตอนที่ 8 ให้ครอบคลุมถนนทุกระดับ แล้วรันใหม่ตั้งแต่ขั้นตอนที่ 8")
else:
    G_mtr = G_proj.edge_subgraph([(u, v, k) for u, v, k in mtr_edges]).copy()
    nearest_edges_mtr, dists_m_mtr = ox.distance.nearest_edges(G_mtr, X, Y, return_dist=True)
    sites_clean['dist_to_mittraphap_road_km'] = np.array(dists_m_mtr) / 1000
    display(sites_clean[['site_id', 'dist_to_nearest_road_km', 'dist_to_mittraphap_road_km']].sort_values('dist_to_mittraphap_road_km'))


## ขั้นตอนที่ 12: บันทึกผลลัพธ์สุดท้ายและดาวน์โหลด


In [ ]:
output_filename = 'road_distance_osm_verified.csv'
sites_clean.to_csv(output_filename, index=False, encoding='utf-8-sig')
files.download(output_filename)
print(f"บันทึกไฟล์: {output_filename}")


## หมายเหตุสำคัญ

- พิกัดที่ดึงจากลิงก์ Google Maps ในส่วนที่ 1 **ควรสุ่มตรวจสอบด้วยตาอย่างน้อย 3-5 จุด** เทียบกับความรู้ภาคสนามจริง (เช่น จุดที่ทราบว่าอยู่ริมถนน/ใกล้โรงงาน) ก่อนนำไปใช้ในรายงานฉบับสมบูรณ์
- ระยะทางถนนที่คำนวณได้เป็นแบบ **straight-line ถึง edge ที่ใกล้ที่สุด** ไม่ใช่ระยะทางตามถนนจริง สอดคล้องกับวิธี haversine เดิมที่ใช้ใน manuscript (Section 2.9) เพื่อให้เทียบผลกันได้ตรงไปตรงมา
- หากลิงก์ `goo.gl` จำนวนมากดึงพิกัดไม่สำเร็จ (Google ทยอยปิดระบบนี้) ทางเลือกสำรองคือเปิดไฟล์ Google My Maps ต้นฉบับ (ถ้าเคยบันทึกเป็นรายการที่ save ไว้ใน Google Maps) แล้ว export เป็น KML ซึ่งจะมีพิกัดครบทุกจุดในไฟล์เดียว
